## Étape 4.1 : lecture des données

Le CSV est lu depuis HDFS. On vérifie le nombre de colonnes (24 attendues) et de lignes (environ 200 000 attendues).

In [ ]:
import os
from pyspark.sql import SparkSession

DATA_PATH = os.environ.get(
    "DATA_PATH",
    "hdfs://localhost:9000/user/miguel/projet_3IABD/fraudTest.csv",
)

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("fraude")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Lecture du CSV depuis HDFS.
raw = spark.read.csv(DATA_PATH, header=True, inferSchema=False)

# Contrôles : 23 colonnes attendues (dont l'index _c0), environ 555 719 lignes attendues
print("Nombre de colonnes :", len(raw.columns))
print("Nombre de lignes :", raw.count())

raw.printSchema()

26/11/19 16:41:28 WARN Utils: Your hostname, LAPTOP-2O6PEQV6 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/19 16:41:28 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 16:41:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Nombre de colonnes : 23


Nombre de lignes : 555719
root
 |-- _c0: string (nullable = true)
 |-- trans_date_trans_time: string (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: string (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (nullable = true)
 |-- city_pop: string (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: string (nullable = true)
 |-- merch_lat: string (nullable = true)
 |-- merch_long: string (nullable = true)
 |-- is_fraud: string (nullable = true)



## Étape 4.2 : renommage et typage des colonnes

Les colonnes sont renommées en français, puis typées : les montants, l'âge et la cible en nombres, la date en type date, le reste en texte.

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, LongType

# Correspondance entre les noms (la colonne d'index "_c0" est ignorée automatiquement)
RENOMMAGE = {
    "trans_date_trans_time": "date_heure_transaction",
    "cc_num": "num_carte",
    "merchant": "marchand",
    "category": "categorie_marchand",
    "amt": "montant",
    "first": "prenom_client",
    "last": "nom_client",
    "gender": "genre",
    "street": "rue_client",
    "city": "ville",
    "state": "etat",
    "zip": "code_postal",
    "lat": "lat_client",
    "long": "long_client",
    "city_pop": "population_ville",
    "job": "profession",
    "dob": "date_naissance",
    "trans_num": "id_transaction",
    "unix_time": "timestamp_unix",
    "merch_lat": "lat_marchand",
    "merch_long": "long_marchand",
    "is_fraud": "est_fraude",
}

# Contrôle : toutes les colonnes attendues doivent exister dans le fichier.
manquantes = [c for c in RENOMMAGE if c not in raw.columns]
if manquantes:
    raise ValueError(f"Colonnes absentes du fichier : {manquantes}")

df = raw.select([F.col(c).alias(RENOMMAGE[c]) for c in RENOMMAGE])

# Typage des colonnes
df = (
    df
    .withColumn("date_heure_transaction", F.to_timestamp(F.col("date_heure_transaction"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("date_naissance", F.to_date(F.col("date_naissance"), "yyyy-MM-dd"))
    .withColumn("montant", F.col("montant").cast(DoubleType()))
    .withColumn("lat_client", F.col("lat_client").cast(DoubleType()))
    .withColumn("long_client", F.col("long_client").cast(DoubleType()))
    .withColumn("lat_marchand", F.col("lat_marchand").cast(DoubleType()))
    .withColumn("long_marchand", F.col("long_marchand").cast(DoubleType()))
    .withColumn("population_ville", F.col("population_ville").cast(IntegerType()))
    .withColumn("timestamp_unix", F.col("timestamp_unix").cast(LongType()))
    .withColumn("est_fraude", F.col("est_fraude").cast(IntegerType()))
)

# Contrôle : une valeur qui ne se convertit pas devient null.
cols_typees = [
    "date_heure_transaction", "date_naissance", "montant",
    "lat_client", "long_client", "lat_marchand", "long_marchand",
    "population_ville", "timestamp_unix", "est_fraude"
]
for c in cols_typees:
    n = df.filter(F.col(c).isNull()).count()
    print(f"{c} : {n} valeur(s) non convertie(s)")

df.printSchema()

date_heure_transaction : 0 valeur(s) non convertie(s)


date_naissance : 0 valeur(s) non convertie(s)


montant : 0 valeur(s) non convertie(s)
lat_client : 0 valeur(s) non convertie(s)
long_client : 0 valeur(s) non convertie(s)
lat_marchand : 0 valeur(s) non convertie(s)
long_marchand : 0 valeur(s) non convertie(s)
population_ville : 0 valeur(s) non convertie(s)
timestamp_unix : 0 valeur(s) non convertie(s)
est_fraude : 0 valeur(s) non convertie(s)
root
 |-- date_heure_transaction: timestamp (nullable = true)
 |-- num_carte: string (nullable = true)
 |-- marchand: string (nullable = true)
 |-- categorie_marchand: string (nullable = true)
 |-- montant: double (nullable = true)
 |-- prenom_client: string (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- rue_client: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- etat: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- lat_client: double (nullable = true)
 |-- long_client: double (nullable = true)
 |-- population_ville: integer (nullable = true)
 |-

## Étape 4.3 : statistiques descriptives

On décrit les colonnes numériques, la répartition de la cible `est_fraude`, puis les liens entre les variables et la fraude. Lors d'une analyse préalable sur le fichier complet, le taux de fraude était d'environ 5 % et les corrélations proches de 0. Ces chiffres devront être retrouvés ici, et interprétés après exécution.

In [3]:
from pyspark.sql import functions as F

colonnes_num = ["montant", "population_ville", "est_fraude"]

df.select(colonnes_num).describe().show()

# 1. Corrélation de chaque variable numérique avec la cible
for c in ["montant", "population_ville"]:
    r = df.stat.corr(c, "est_fraude")
    print(f"corrélation {c} / est_fraude : {r:.4f}")

# 2. Nombre de transactions normales et frauduleuses
total = df.count()
(
    df.groupBy("est_fraude").count()
    .withColumn("taux_%", F.round(F.col("count") / total * 100, 2))
    .orderBy("est_fraude")
    .show()
)

# 3. Montant moyen par classe
df.groupBy("est_fraude").agg(F.round(F.avg("montant"), 2).alias("montant_moyen")).show()

# 4. Taux de fraude par catégorie
for c in ["categorie_marchand", "genre"]:
    (
        df.groupBy(c)
        .agg(F.round(F.avg("est_fraude") * 100, 2).alias("taux_fraude_%"))
        .orderBy(F.desc("taux_fraude_%"))
        .show()
    )

# 5. Table croisée pour le Heatmap : taux de fraude (%) par catégorie et par heure
heatmap_df = (
    df.withColumn("heure", F.hour("date_heure_transaction"))
    .groupBy("categorie_marchand")
    .pivot("heure", list(range(24)))
    .agg(F.round(F.avg("est_fraude") * 100, 2))
    .fillna(0.0)
)
heatmap_df.toPandas().to_csv(f"{OUT}/heatmap_fraude.csv", index=False)
print("Heatmap exporté dans", f"{OUT}/heatmap_fraude.csv")

# 6. Export des statistiques descriptives
stats_pd = df.select(colonnes_num).describe().toPandas()
stats_pd.to_csv(f"{OUT}/statistiques.csv", index=False)
print("Statistiques exportées dans", f"{OUT}/statistiques.csv")

+-------+------------------+------------------+--------------------+
|summary|           montant|  population_ville|          est_fraude|
+-------+------------------+------------------+--------------------+
|  count|            555719|            555719|              555719|
|   mean| 69.39281023322945| 88221.88791817447|0.003859864427885...|
| stddev|156.74594135531495|300390.89206099446|0.062007844611783956|
|    min|               1.0|                23|                   0|
|    max|          22768.11|           2906700|                   1|
+-------+------------------+------------------+--------------------+



corrélation montant / est_fraude : 0.1823
corrélation population_ville / est_fraude : -0.0049


+----------+------+------+
|est_fraude| count|taux_%|
+----------+------+------+
|         0|553574| 99.61|
|         1|  2145|  0.39|
+----------+------+------+



+----------+-------------+
|est_fraude|montant_moyen|
+----------+-------------+
|         1|       528.36|
|         0|        67.61|
+----------+-------------+

+------------------+-------------+
|categorie_marchand|taux_fraude_%|
+------------------+-------------+
|      shopping_net|         1.21|
|          misc_net|         0.98|
|       grocery_pos|         0.92|
|      shopping_pos|         0.43|
|     gas_transport|         0.27|
|            travel|         0.23|
|       grocery_net|         0.21|
|          misc_pos|         0.21|
|     personal_care|         0.18|
|     entertainment|         0.15|
|       food_dining|         0.14|
|    health_fitness|         0.14|
|         kids_pets|         0.13|
|              home|         0.13|
+------------------+-------------+

+-----+-------------+
|genre|taux_fraude_%|
+-----+-------------+
|    M|         0.39|
|    F|         0.38|
+-----+-------------+



26/11/19 16:41:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Heatmap exporté dans outputs/heatmap_fraude.csv


Statistiques exportées dans outputs/statistiques.csv


## Étape 4.4 : nettoyage

Le nettoyage porte sur cinq points : suppression de la colonne constante `devise`, retrait des données personnelles, doublons sur l'identifiant de transaction, valeurs manquantes, et plages de valeurs. Lors de l'analyse préalable, aucun doublon ni valeur manquante n'a été trouvé : la cellule le vérifie et le documente.

In [4]:
from pyspark.sql import functions as F

# 1. Colonne redondante (timestamp_unix fait doublon avec date_heure_transaction)
df = df.drop("timestamp_unix")

# 2. Doublons (AVANT de supprimer id_transaction)
n_avant = df.count()
df = df.dropDuplicates(["id_transaction"])
print("Doublons supprimés :", n_avant - df.count())

# 3. Données personnelles et identifiants uniques
df = df.drop("prenom_client", "nom_client", "rue_client", "num_carte", "id_transaction")

# 4. Valeurs manquantes
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

# Suppression des lignes incomplètes
n_avant = df.count()
df = df.dropna()
print("Lignes incomplètes supprimées :", n_avant - df.count())

# 5. Retire les lignes incohérentes
n_avant = df.count()
df = df.filter(
    (F.col("montant") > 0)
    & (F.col("population_ville") > 0)
    & F.col("lat_client").between(-90, 90)
    & F.col("long_client").between(-180, 180)
    & F.col("est_fraude").isin(0, 1)
)
print("Lignes incohérentes supprimées :", n_avant - df.count())

# Nombre de lignes et de colonnes conservées
print("Lignes après nettoyage :", df.count())
print("Colonnes après nettoyage :", len(df.columns))

Doublons supprimés : 0


+----------------------+--------+------------------+-------+-----+-----+----+-----------+----------+-----------+----------------+----------+--------------+------------+-------------+----------+
|date_heure_transaction|marchand|categorie_marchand|montant|genre|ville|etat|code_postal|lat_client|long_client|population_ville|profession|date_naissance|lat_marchand|long_marchand|est_fraude|
+----------------------+--------+------------------+-------+-----+-----+----+-----------+----------+-----------+----------------+----------+--------------+------------+-------------+----------+
|0                     |0       |0                 |0      |0    |0    |0   |0          |0         |0          |0               |0         |0             |0           |0            |0         |
+----------------------+--------+------------------+-------+-----+-----+----+-----------+----------+-----------+----------------+----------+--------------+------------+-------------+----------+



Lignes incomplètes supprimées : 0


Lignes incohérentes supprimées : 0


Lignes après nettoyage : 555719
Colonnes après nettoyage : 16


## Étape 4.5 : features

Les modèles ne lisent que des nombres. Cette étape extrait l'heure et le jour de la semaine, encode les colonnes texte, puis regroupe toutes les variables dans un vecteur `features`. Seules les colonnes texte à peu de valeurs distinctes sont retenues ; les colonnes à très nombreuses valeurs (ville, état, agence, localisation, appareil de transaction) sont écartées pour garder un modèle simple.

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
import pandas as pd
from pyspark.ml.stat import Correlation

# 1. Feature engineering : heure, jour de la semaine, âge du client et distance client-marchand (km)
df_feat = (
    df
    .withColumn("heure", F.hour(F.col("date_heure_transaction")))
    .withColumn("jour_semaine", F.dayofweek(F.col("date_heure_transaction")))
    .withColumn(
        "age",
        F.floor(F.datediff(F.col("date_heure_transaction"), F.col("date_naissance")) / 365.25).cast(IntegerType())
    )
    .withColumn(
        "distance_km",
        F.sqrt(
            F.pow(F.col("lat_client") - F.col("lat_marchand"), 2)
            + F.pow(F.col("long_client") - F.col("long_marchand"), 2)
        ) * 111.0
    )
)

# 2. Gestion du déséquilibre (~0.39% de fraude) via une colonne de poids
total = df_feat.count()
nb_fraudes = df_feat.filter(F.col("est_fraude") == 1).count()
nb_normales = total - nb_fraudes
poids_fraude = nb_normales / nb_fraudes  # ~250x plus de poids sur la classe fraude

df_feat = df_feat.withColumn(
    "poids",
    F.when(F.col("est_fraude") == 1, F.lit(poids_fraude)).otherwise(F.lit(1.0))
)


# 3. Calcul et export de la matrice de corrélation (variables numériques + feature engineering)
cols_corr = ["montant", "population_ville", "age", "heure", "jour_semaine", "distance_km", "est_fraude"]

vec_corr = (
    VectorAssembler(inputCols=cols_corr, outputCol="features_corr")
    .transform(df_feat)
    .select("features_corr")
)

matrice_corr = Correlation.corr(vec_corr, "features_corr", method="pearson").head()[0].toArray()
corr_pd = pd.DataFrame(matrice_corr, index=cols_corr, columns=cols_corr).round(4)

corr_pd.to_csv(f"{OUT}/correlation.csv")
print(f"Matrice de corrélation exportée dans {OUT}/correlation.csv")

# 4. Séparation entraînement (80%) / test (20%) avant le Pipeline
train_raw, test_raw = df_feat.randomSplit([0.8, 0.2], seed=42)

# 5. Définition des variables utiles (catégorielles + numériques)
cols_cat = ["categorie_marchand", "genre", "etat"]
cols_num = ["montant", "population_ville", "age", "heure", "jour_semaine", "distance_km"]

# 6. Construction du Pipeline de prétraitement
indexeurs = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cols_cat]
encodeurs = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_ohe"]) for c in cols_cat]

entrees = [f"{c}_ohe" for c in cols_cat] + cols_num
assembleur = VectorAssembler(inputCols=entrees, outputCol="features_raw")
normalisateur = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

pipeline = Pipeline(stages=indexeurs + encodeurs + [assembleur, normalisateur])

# Ajustement sur train uniquement, puis transformation sur train et test
modele_prep = pipeline.fit(train_raw)
train_df = modele_prep.transform(train_raw).select("features", "est_fraude", "poids")
test_df = modele_prep.transform(test_raw).select("features", "est_fraude", "poids")

print(f"Poids fraude : {poids_fraude:.1f} | Train : {train_df.count()} lignes | Test : {test_df.count()} lignes")

Matrice de corrélation exportée dans outputs/correlation.csv


Poids fraude : 258.1 | Train : 444928 lignes | Test : 110791 lignes


## Étape 4.6 : Machine Learning


.

In [ ]:
import pandas as pd
from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    MultilayerPerceptronClassifier,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)

# 1. Préparation spécifique pour le MLP (ne supporte pas weightCol -> rééquilibrage du train uniquement)
nb_features = train_df.first()["features"].size
train_mlp = train_df.sampleBy("est_fraude", fractions={0: 0.03, 1: 1.0}, seed=42).cache()
print(f"Features en entrée : {nb_features} | Lignes train MLP : {train_mlp.count()}")

# 2. Définition des trois modèles et de leur jeu d'entraînement respectif
modeles = {
    "LogisticRegression": (
        LogisticRegression(
            featuresCol="features",
            labelCol="est_fraude",
            weightCol="poids",
            maxIter=50,
        ),
        train_df,
    ),
    "RandomForest": (
        RandomForestClassifier(
            featuresCol="features",
            labelCol="est_fraude",
            weightCol="poids",
            numTrees=100,
            maxDepth=7,
            seed=42,
        ),
        train_df,
    ),
    "MLP (Perceptron)": (
        MultilayerPerceptronClassifier(
            featuresCol="features",
            labelCol="est_fraude",
            layers=[nb_features, 32, 16, 2],
            maxIter=50,
            blockSize=128,
            seed=42,
        ),
        train_mlp,
    ),
}

# 3. Évaluateurs communs (tous les modèles sont testés sur le MÊME test_df non modifié)
eval_pr = BinaryClassificationEvaluator(
    labelCol="est_fraude", rawPredictionCol="rawPrediction", metricName="areaUnderPR"
)
eval_roc = BinaryClassificationEvaluator(
    labelCol="est_fraude", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
eval_f1_fraude = MulticlassClassificationEvaluator(
    labelCol="est_fraude",
    predictionCol="prediction",
    metricName="fMeasureByLabel",
    metricLabel=1.0,
)

resultats = []

# 4. Entraînement et évaluation de chaque modèle
for nom, (algo, data_train) in modeles.items():
    print(f"\n=== Entraînement : {nom} ===")
    modele = algo.fit(data_train)
    preds = modele.transform(test_df)

    auprc = eval_pr.evaluate(preds)
    auroc = eval_roc.evaluate(preds)
    f1_fraude = eval_f1_fraude.evaluate(preds)

    # Extraction de la matrice de confusion en une seule passe (économise la RAM sur WSL)
    cm_rows = preds.groupBy("est_fraude", "prediction").count().collect()
    cm = {(int(r["est_fraude"]), int(r["prediction"])): r["count"] for r in cm_rows}

    vn = cm.get((0, 0), 0)
    fp = cm.get((0, 1), 0)
    fn = cm.get((1, 0), 0)
    vp = cm.get((1, 1), 0)

    rappel = (vp / (vp + fn) * 100) if (vp + fn) > 0 else 0.0
    precision = (vp / (vp + fp) * 100) if (vp + fp) > 0 else 0.0

    resultats.append({
        "modele": nom,
        "AUPRC": round(auprc, 4),
        "AUROC": round(auroc, 4),
        "F1_Fraude": round(f1_fraude, 4),
        "Fraudes_detectees": vp,
        "Fraudes_manquees": fn,
        "Fausses_alarmes": fp,
        "Rappel_%": round(rappel, 2),
        "Precision_%": round(precision, 2),
    })

    print(f"AUPRC : {auprc:.4f} | AUROC : {auroc:.4f} | F1_Fraude : {f1_fraude:.4f}")
    print(f"Matrice -> VN: {vn} | FP (fausses alarmes): {fp} | FN (manquées): {fn} | VP (détectées): {vp}")

# Libère le petit cache du MLP
train_mlp.unpersist()

# 5. Tableau comparatif et export pour le dashboard (app.py)
res_df = pd.DataFrame(resultats)
print("\n=== Comparaison des modèles ===")
print(res_df.to_string(index=False))

res_df.to_csv(f"{OUT}/resultats_modeles.csv", index=False)
print(f"\nRésultats exportés dans {OUT}/resultats_modeles.csv")


=== Entraînement : LogisticRegression ===


26/11/19 16:42:51 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/19 16:42:51 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS


AUPRC : 0.1182 | AUROC : 0.9175 | F1_Fraude : 0.0475
Matrice de confusion :


+----------+----------+-----+
|est_fraude|prediction|count|
+----------+----------+-----+
|         0|       0.0|97578|
|         0|       1.0|12931|
|         1|       0.0|   94|
|         1|       1.0|  325|
+----------+----------+-----+


=== Entraînement : RandomForest ===


26/11/19 16:44:20 WARN DAGScheduler: Broadcasting large task binary with size 1074.0 KiB


AUPRC : 0.6366 | AUROC : 0.9810 | F1_Fraude : 0.1814
Matrice de confusion :


+----------+----------+------+
|est_fraude|prediction| count|
+----------+----------+------+
|         0|       0.0|107319|
|         0|       1.0|  3190|
|         1|       0.0|    59|
|         1|       1.0|   360|
+----------+----------+------+


=== Comparaison des modèles ===
            modele  AUPRC  AUROC  F1_Fraude
LogisticRegression 0.1182 0.9175     0.0475
      RandomForest 0.6366 0.9810     0.1814

Résultats exportés dans outputs/resultats_modeles.csv
